# Portfolio Construction Research

**Maintained continuation of the 2024 CQF portfolio-construction study**

This is the repository's authoritative analysis. It develops the original study into a reproducible portfolio research workflow covering Black–Litterman views, minimum-variance and tracking objectives, higher-moment utility, fractional Kelly allocation, beta-neutral betting-against-beta construction, optional Fama–French five-factor regression, chronological walk-forward evaluation, turnover, transaction costs and short-borrow costs.

The original August 2024 notebook is preserved in `research_archive/portfolio_construction_2024.ipynb` as provenance. It is not a second implementation and is not required to run this analysis. Corrections from the original study are declared in `AMENDMENTS.md`.

Run all cells from top to bottom. The standard research setting requests adjusted Yahoo prices. For deterministic offline validation, set `DATA_MODE = "synthetic"`.


## 1. Research configuration

- `DATA_MODE`: `"yahoo"` for adjusted historical prices or `"synthetic"` for deterministic validation data.
- Dates use `YYYY-MM-DD`; the end date is exclusive.
- `TRAINING_DAYS` is the minimum expanding-window history before evaluation.
- Targets are refitted at the first available observation of each calendar month.
- `TRANSACTION_COST_BPS` is charged per unit of one-way turnover.
- `ANNUAL_SHORT_BORROW_BPS` is charged daily against gross short exposure.
- `KELLY_FRACTION` is the risky allocation; the remainder stays in cash.
- `BAB_GROSS_LIMIT` bounds absolute long-plus-short exposure.
- `FACTOR_FILE` is optional. When supplied, `FACTOR_UNIT` must explicitly state whether its values are percentage points or decimals.


In [ ]:
DATA_MODE = "yahoo"
START_DATE = "2020-01-01"
END_DATE = "2024-08-23"
ANNUAL_RISK_FREE_RATE = 0.03
MAX_WEIGHT = 0.35
TRAINING_DAYS = 504
TRANSACTION_COST_BPS = 10
ANNUAL_SHORT_BORROW_BPS = 100
KELLY_FRACTION = 0.50
BAB_GROSS_LIMIT = 2.0
FACTOR_FILE = "data/F-F_Research_Data_5_Factors_2x3_monthly.csv"
FACTOR_UNIT = "percent"
RANDOM_SEED = 42


## 2. Load the research modules

`portfolio_tools.py` contains the data, Black–Litterman, optimization, walk-forward and reporting workflow. `research_extensions.py` contains the factor, higher-moment, Kelly and BAB methods. Both modules are covered by behavioral tests.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from portfolio_tools import ASSETS, BENCHMARK, load_prices, analyze, charts, save_run
from research_extensions import factor_regression, load_fama_french_csv, monthly_compound

pd.set_option("display.max_columns", 30)
print("Research modules ready. Data mode:", DATA_MODE)


## 3. Acquire and validate prices

The analysis uses a date-by-symbol adjusted-price table, so returns are calculated within each asset and never across symbol boundaries. Yahoo failures stop the run instead of substituting synthetic observations. The source mode, requested dates, coverage and price-table SHA-256 hash are recorded with every output.


In [ ]:
prices, provenance = load_prices(DATA_MODE, START_DATE, END_DATE, RANDOM_SEED)
print(provenance["label"])
print(f"{len(prices):,} price rows, {prices.shape[1]} asset labels")
print("First date:", provenance["first_price"], "| Last date:", provenance["last_price"])
display(prices.head().round(2))

## 4. Fit and evaluate the portfolio methods

All strategies follow the same chronological contract. After the initial training history, each monthly target uses only observations strictly before its rebalance date. Asset weights drift between rebalances. The simulation records one-way turnover and deducts the stated trading costs; short portfolios also incur the stated borrow cost.

| Portfolio | Maintained objective |
| --- | --- |
| Equal weight | Equal long-only asset targets |
| Minimum variance | Minimum annualized covariance-based variance |
| BL maximum Sharpe | Maximum expected Sharpe using the Black–Litterman posterior |
| Minimum tracking error | Minimum annualized active-return variance versus SPY |
| Higher-moment utility | Explicit annual mean/variance utility with sample skew reward and excess-kurtosis penalty |
| Fractional Kelly | Maximum sample log growth for a fractional risky sleeve plus cash |
| BAB beta-neutral | Low-beta long/high-beta short legs scaled to zero estimated market beta and bounded gross exposure |

Covariance and correlation remain separate. Expected returns and covariance use annual units; daily performance statistics use the documented 252-observation convention.


In [ ]:
result = analyze(
    prices,
    annual_rf=ANNUAL_RISK_FREE_RATE,
    max_weight=MAX_WEIGHT,
    training_days=TRAINING_DAYS,
    transaction_cost_bps=TRANSACTION_COST_BPS,
    annual_short_borrow_bps=ANNUAL_SHORT_BORROW_BPS,
    kelly_fraction=KELLY_FRACTION,
    bab_gross_limit=BAB_GROSS_LIMIT,
)
print("Aligned returns:", len(result["returns"]))
print("Training observations:", result["training_days"])
print("Walk-forward evaluation:", result["evaluation_start"].date(), "to", result["evaluation_end"].date())
display(result["returns"][ASSETS].corr().round(2))


## 5. Black–Litterman views

The three views are retained as explicit August 2024 research scenarios: GLD over MTUM by 15 percentage points annually, XLK over XLE by 8 points, and SPHD over VUG by 6 points. They are assumptions, not current forecasts.

The prior is $\Pi=\delta\Sigma w_0$. View uncertainty is scaled from $P(	au\Sigma)P^T$ and the stated confidence. The table reports annual excess-return estimates in consistent decimal units.


In [ ]:
display(result["views"].style.format({"Annual difference": "{:.1%}", "Assumed confidence": "{:.0%}"}))
display(pd.DataFrame({"Prior excess return": result["prior"],
                      "BL posterior excess return": result["posterior"]}).style.format("{:.2%}"))

## 6. Latest portfolio targets

The table includes risky assets and cash. Long-only methods sum to 100%; fractional Kelly deliberately retains cash. BAB may contain negative asset weights and an offsetting cash position because it is a beta-neutral long/short construction. The full target history is exported for audit.


In [ ]:
print(provenance["label"])
display(result["weights"].style.format("{:.1%}"))
print("Budget sums:", result["weights"].sum().round(8).to_dict())
print("Latest BAB diagnostics:")
display(result["bab_diagnostics"].style.format({"Beta": "{:.3f}", "Weight": "{:.2%}"}))
print("BAB estimated beta:", round(result["bab_diagnostics"].attrs["portfolio_beta"], 10))
print("BAB gross exposure:", round(result["bab_diagnostics"].attrs["gross_exposure"], 4))


## 7. Model, implementation and walk-forward results

The model table describes the final pre-trade estimates. The implementation table aggregates one-way turnover and modeled costs. Historical statistics use only later walk-forward returns and include transaction and short-borrow costs for portfolio strategies; SPY is an uncosted reference.

Walk-forward separation reduces look-ahead bias but does not create independent market evidence. The views are fixed scenarios, parameters were not selected through nested validation, and the cost model does not include market impact, taxes or locate constraints.


In [ ]:
print("LATEST MODEL ESTIMATES —", provenance["label"])
display(result["model"].style.format({"Expected annual return": "{:.2%}",
                                      "Annual volatility": "{:.2%}", "Expected Sharpe": "{:.2f}"}))
print("IMPLEMENTATION —", TRANSACTION_COST_BPS, "bps transaction cost;",
      ANNUAL_SHORT_BORROW_BPS, "bps annual short borrow")
display(result["trading"].style.format({"One-way turnover": "{:.2f}",
                                        "Estimated cost": "{:.2%}",
                                        "Estimated short-borrow cost": "{:.2%}",
                                        "Estimated total implementation cost": "{:.2%}",
                                        "Annualized one-way turnover": "{:.2f}",
                                        "Cost assumption (bps)": "{:.0f}"}))
print("NET WALK-FORWARD HISTORY —", provenance["label"])
display(result["historical"].style.format({"Annual mean": "{:.2%}", "CAGR": "{:.2%}",
                                           "Annual volatility": "{:.2%}", "Sharpe": "{:.2f}",
                                           "Max drawdown": "{:.2%}"}))


## 8. Optional Fama–French five-factor attribution

If the cleaned monthly factor file exists, this section converts it from the explicitly declared unit, compounds daily portfolio returns to months, aligns dates, and regresses portfolio excess returns on Mkt-RF, SMB, HML, RMW and CMA. It reports annualized alpha, loadings, t-statistics, R-squared and sample size.

The factor file is not redistributed. Its preparation contract is documented in `data/README.md`. A missing file skips attribution without changing portfolio results.


In [ ]:
factor_path = Path(FACTOR_FILE)
factor_results = None
if factor_path.exists():
    factors = load_fama_french_csv(factor_path, unit=FACTOR_UNIT)
    monthly_portfolios = monthly_compound(result["portfolio_returns"].drop(columns=[BENCHMARK]))
    factor_results = factor_regression(monthly_portfolios, factors)
    display(factor_results.style.format({"Annual alpha": "{:.2%}", "Alpha t-stat": "{:.2f}",
                                         "R-squared": "{:.2f}"}))
else:
    print("Factor attribution skipped; file not found:", factor_path)
    print("See data/README.md for the required schema and explicit unit setting.")


## 9. Portfolio charts

The allocation chart includes cash and signed BAB weights. The growth chart covers only the later walk-forward period. Portfolio series are net of the stated modeled costs; SPY is an uncosted reference.


In [ ]:
weight_chart, growth_chart = charts(result, provenance["label"])
plt.show()

## 10. Save the research outputs

Each run creates a new timestamped folder under `outputs/`. It contains latest targets, full target history, rebalance turnover, implementation summaries, metrics, charts and `run_info.json` with data provenance and methodology settings. When available, factor attribution is saved beside them.


In [ ]:
output_folder = save_run(result, provenance, ANNUAL_RISK_FREE_RATE, MAX_WEIGHT)
if factor_results is not None:
    factor_results.to_csv(output_folder / "factor_regression.csv")
weight_chart.savefig(output_folder / "allocations.png", dpi=150, bbox_inches="tight")
growth_chart.savefig(output_folder / "growth.png", dpi=150, bbox_inches="tight")
print("RUN COMPLETE")
print("Data:", provenance["label"])
print("Saved results:", output_folder.resolve())


## Research interpretation and next work

This implementation makes the portfolio methods and evaluation contract explicit and testable. It does not treat the best historical Sharpe ratio as proof of superiority. Useful next work includes multiple non-overlapping evaluation periods, parameter stability maps, block-bootstrap uncertainty, alternative view-generation processes, point-in-time factor vintages and richer execution-cost models.

### References

- Black, F. and Litterman, R. (1992), *Global Portfolio Optimization*, Financial Analysts Journal, 48(5), 28–43.
- Fama, E. F. and French, K. R. (2015), *A Five-Factor Asset Pricing Model*, Journal of Financial Economics, 116(1), 1–22.
- Frazzini, A. and Pedersen, L. H. (2014), *Betting Against Beta*, Journal of Financial Economics, 111(1), 1–25.
- Kelly, J. L. (1956), *A New Interpretation of Information Rate*, Bell System Technical Journal, 35(4), 917–926.
- `AMENDMENTS.md` documents corrections from the 2024 implementation.

This repository is quantitative research code, not personalized investment advice or evidence of future returns.
